# Document & Document Loaders (2026년 최신 권장 사용법)

> **⚠️ 2026년 9월 기준 변경 사항 (공통)**
>
> - 책에서 사용한 `langchain_community.document_loaders` 는 **`langchain-community` 패키지 sunset**(2026년 5월 발표, 6월 저장소 아카이브)으로 더 이상 유지보수되지 않습니다. 설치·import 는 여전히 되지만 새 프로젝트의 기반으로 권장되지 않습니다.
> - LangChain 의 현재 방향은 ① **전용 통합 패키지**(`langchain-upstage`, `langchain-unstructured`, `langchain-pymupdf4llm`, `langchain-docling` 등)를 쓰거나, ② 파싱 라이브러리를 **직접 사용**하고 결과를 `langchain_core.documents.Document` 로 감싸는 것(필요하면 `BaseLoader` 를 상속한 작은 로더 클래스 작성)입니다.
> - `Document`, `BaseLoader`, 텍스트 분할기(`langchain-text-splitters`)는 그대로 유지되는 핵심 인터페이스입니다.

**이 노트북에서 바뀐 점**

| 책(구버전) | 현재 권장 |
|---|---|
| `langchain_community.document_loaders.PyPDFLoader` | `BaseLoader` 를 상속한 커스텀 로더(`pypdf` 직접 사용) 또는 전용 패키지 `langchain-pymupdf4llm` |
| `document.__dict__` | `document.model_dump()` (Pydantic v2 모델) |
| `loader.load_and_split(text_splitter)` | `text_splitter.split_documents(loader.load())` — `load_and_split` 은 사실상 deprecated |
| `adocs = loader.aload()` → `await adocs` | `docs = await loader.aload()` / `async for doc in loader.alazy_load()` |

**참고**

- [LangChain Document loaders 개요](https://docs.langchain.com/oss/python/integrations/document_loaders)
- [`BaseLoader` API 레퍼런스](https://reference.langchain.com/python/langchain-core/document_loaders/base/BaseLoader)

In [ ]:
# 설치
# !pip install -qU langchain-core langchain-text-splitters pypdf

## 실습에 활용한 문서

소프트웨어정책연구소(SPRi) - 2023년 12월호

- 저자: 유재흥(AI정책연구실 책임연구원), 이지수(AI정책연구실 위촉연구원)
- 링크: https://spri.kr/posts/view/23669
- 파일명: `SPRI_AI_Brief_2023년12월호_F.pdf`

## Document

LangChain 의 기본 문서 객체입니다. (`langchain_core` 에 있으므로 community sunset 과 무관합니다.)

**속성**
- `page_content`: 문서의 내용을 나타내는 문자열입니다.
- `metadata`: 문서의 메타데이터를 나타내는 딕셔너리입니다.
- `id`: (선택) 문서의 고유 식별자입니다. 벡터스토어에 저장할 때 중복 방지/갱신에 활용됩니다.

In [ ]:
from langchain_core.documents import Document

document = Document(
    page_content="안녕하세요? 이건 랭체인의 도큐먼트 입니다",
    metadata={"source": "TeddyNote"},  # 생성 시점에 메타데이터 지정 가능
    id="doc-001",  # 선택 사항
)

In [ ]:
# 도큐먼트의 속성 확인 (Document 는 Pydantic v2 모델이므로 model_dump() 사용)
document.model_dump()

metadata 에 속성 추가

In [ ]:
# 메타데이터 추가
document.metadata["page"] = 1
document.metadata["author"] = "Teddy"

# 여러 개를 한 번에 추가할 때는 update 도 편리합니다.
document.metadata.update({"lang": "ko"})

In [ ]:
# 도큐먼트의 메타데이터 확인
document.metadata

## Document Loader

다양한 형식의 파일/소스에서 불러온 내용을 `Document` 객체로 변환하는 역할을 합니다.

모든 로더는 `langchain_core.document_loaders.BaseLoader` 인터페이스를 따릅니다.

| 메서드 | 설명 |
|---|---|
| `lazy_load()` | **구현해야 하는 핵심 메서드.** generator 로 문서를 하나씩 반환 |
| `load()` | `list(self.lazy_load())` — 전체를 한 번에 로드 |
| `alazy_load()` | 비동기 generator (기본 구현은 `lazy_load` 를 스레드에서 실행) |
| `aload()` | 비동기 전체 로드 |

### 주요 로더 (2026년 기준 권장 대안)
- PDF: `PyMuPDF4LLMLoader`(langchain-pymupdf4llm), `UpstageDocumentParseLoader`(langchain-upstage), `DoclingLoader`(langchain-docling), `UnstructuredLoader`(langchain-unstructured), 또는 `pypdf`/`pdfplumber` 직접 사용
- CSV / Excel: `csv`·`pandas` 직접 사용 + `Document` 변환, 또는 `UnstructuredLoader`
- HTML / 웹: `httpx` + `BeautifulSoup` 직접 사용, 또는 `UnstructuredLoader`, `DoclingLoader`
- JSON: `json`(필요 시 `jq`) 직접 사용
- 텍스트 / 디렉토리: `pathlib` 직접 사용

이 노트북에서는 인터페이스를 이해하기 위해 **`pypdf` 기반의 PDF 로더를 직접 구현**해 봅니다.

In [ ]:
# 예제 파일 경로
FILE_PATH = "./data/SPRI_AI_Brief_2023년12월호_F.pdf"

In [ ]:
from pathlib import Path
from typing import Iterator

from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document
from pypdf import PdfReader


class PDFPageLoader(BaseLoader):
    """pypdf 로 PDF 를 페이지 단위 Document 로 변환하는 최소 구현 로더"""

    def __init__(self, file_path: str | Path) -> None:
        self.file_path = Path(file_path)

    def lazy_load(self) -> Iterator[Document]:
        reader = PdfReader(self.file_path)
        total_pages = len(reader.pages)
        for page_number, page in enumerate(reader.pages):
            yield Document(
                page_content=page.extract_text() or "",
                metadata={
                    "source": str(self.file_path),
                    "page": page_number,  # 0부터 시작 (책의 PyPDFLoader 와 동일)
                    "total_pages": total_pages,
                },
            )


# 로더 설정
loader = PDFPageLoader(FILE_PATH)

> 💡 직접 구현 대신 **전용 통합 패키지**를 쓰려면 아래처럼 사용할 수 있습니다. (Markdown 형태로 표·제목 구조를 보존해 RAG 에 유리)
>
> ```python
> # !pip install -qU langchain-pymupdf4llm
> from langchain_pymupdf4llm import PyMuPDF4LLMLoader
> loader = PyMuPDF4LLMLoader(FILE_PATH, mode="page")
> ```

### load()

- 문서를 로드하여 반환합니다.
- 반환된 결과는 `list[Document]` 형태입니다.

In [ ]:
# PDF 로드
docs = loader.load()

# 로드된 문서의 수 확인
len(docs)

In [ ]:
# 여섯 번째 문서 확인
docs[5]

### 문서 분할 (구 `load_and_split()`)

- `load_and_split()` 은 `BaseLoader` 에 남아 있지만 **사용하지 말 것(deprecated 취급)** 으로 명시되어 있습니다.
- 로드와 분할을 분리해서, 분할기의 `split_documents()` 를 직접 호출하는 것이 권장 방식입니다.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문자열 분할기 설정
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0)

# 로드 → 분할 (split_documents 는 Iterable 을 받으므로 lazy_load() 를 바로 넘겨도 됩니다)
split_docs = text_splitter.split_documents(loader.lazy_load())

# 분할된 문서의 수 확인
print(f"문서의 길이: {len(split_docs)}")

# 11번째 조각 확인
split_docs[10]

### lazy_load()

- generator 방식으로 문서를 로드합니다. 대용량 문서를 메모리에 모두 올리지 않고 처리할 때 사용합니다.

In [ ]:
loader.lazy_load()  # generator 객체

In [ ]:
# generator 방식으로 문서 로드
for doc in loader.lazy_load():
    print(doc.metadata)

### aload() / alazy_load()

- 비동기(Async) 방식의 문서 로드입니다.
- Jupyter 는 이미 이벤트 루프가 돌고 있으므로 **top-level `await`** 를 바로 쓸 수 있습니다. (`nest_asyncio` 불필요)
- 일반 `.py` 스크립트에서는 `asyncio.run(main())` 형태로 실행합니다.

In [ ]:
# 문서를 async 방식으로 한 번에 로드
adocs = await loader.aload()
len(adocs)

In [ ]:
# async generator 방식
async for doc in loader.alazy_load():
    print(doc.metadata["page"], end=" ")